In [3]:
# ==========================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# ==========================================
# STEP 2: LOAD TRAINING DATA
# ==========================================

columns = (
    ["unit", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

print("Training data shape:", train_df.shape)
train_df.head()

Training data shape: (20631, 26)


,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
# ==========================================
# STEP 3: REMOVE CONSTANT SENSORS
# ==========================================

sensor_columns = [
    col for col in train_df.columns
    if col.startswith("sensor_")
]

sensor_variability = train_df[sensor_columns].nunique()

constant_sensors = sensor_variability[
    sensor_variability == 1
].index.tolist()

train_df_clean = train_df.drop(
    columns=constant_sensors
).copy()

print("Removed sensors:", constant_sensors)
print("Cleaned training shape:", train_df_clean.shape)

Removed sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Cleaned training shape: (20631, 20)


In [6]:
# ==========================================
# STEP 4: CALCULATE REMAINING USEFUL LIFE
# ==========================================

max_cycles = train_df_clean.groupby("unit")["cycle"].max()

train_df_clean["RUL"] = (
    train_df_clean["unit"].map(max_cycles)
    - train_df_clean["cycle"]
)

print(train_df_clean[["unit", "cycle", "RUL"]].head())

   unit  cycle  RUL
0     1      1  191
1     1      2  190
2     1      3  189
3     1      4  188
4     1      5  187


In [7]:
# ==========================================
# STEP 5: APPLY RUL CAPPING
# ==========================================

RUL_CAP = 125

train_df_clean["RUL"] = train_df_clean["RUL"].clip(
    upper=RUL_CAP
)

print(train_df_clean["RUL"].describe())

count    20631.000000
mean        86.829286
std         41.673699
min          0.000000
25%         51.000000
50%        103.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64


In [8]:
# ==========================================
# STEP 6: LOAD TEST DATA AND RUL LABELS
# ==========================================

test_df = pd.read_csv(
    "../data/raw/test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test_rul = pd.read_csv(
    "../data/raw/RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

# Remove the same constant sensors
test_df = test_df.drop(columns=constant_sensors)

# Apply RUL cap
test_rul["RUL"] = test_rul["RUL"].clip(upper=RUL_CAP)

print("Test shape:", test_df.shape)
print("Test engines:", test_df["unit"].nunique())
print("Test RUL labels:", test_rul.shape)

Test shape: (13096, 20)
Test engines: 100
Test RUL labels: (100, 1)


In [9]:
# ==========================================
# STEP 7: ENGINE-LEVEL TRAIN-VALIDATION SPLIT
# ==========================================

engine_ids = train_df_clean["unit"].unique()

train_units, val_units = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=42
)

train_data = train_df_clean[
    train_df_clean["unit"].isin(train_units)
].copy()

val_data = train_df_clean[
    train_df_clean["unit"].isin(val_units)
].copy()

print("Training engines:", train_data["unit"].nunique())
print("Validation engines:", val_data["unit"].nunique())

Training engines: 80
Validation engines: 20


In [10]:
# ==========================================
# STEP 8: FEATURE SCALING
# ==========================================

feature_columns = [
    col for col in train_df_clean.columns
    if col.startswith("setting_") or col.startswith("sensor_")
]

scaler = StandardScaler()

train_data[feature_columns] = scaler.fit_transform(
    train_data[feature_columns]
)

val_data[feature_columns] = scaler.transform(
    val_data[feature_columns]
)

test_df[feature_columns] = scaler.transform(
    test_df[feature_columns]
)

print("Number of features:", len(feature_columns))
print("Scaling completed!")

Number of features: 18
Scaling completed!


In [11]:
# STEP 45: SAVE PREPROCESSING SCALER

import joblib
import os

os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    scaler,
    "../artifacts/scaler.pkl"
)

joblib.dump(
    feature_columns,
    "../artifacts/feature_columns.pkl"
)

print("Scaler and feature columns saved successfully!")

Scaler and feature columns saved successfully!


In [12]:
# ==========================================
# STEP 9: CREATE SLIDING WINDOWS
# ==========================================

WINDOW_SIZE = 30

def create_windows(data, feature_columns, window_size=30):

    X, y = [], []

    for unit in data["unit"].unique():

        engine_data = data[data["unit"] == unit].sort_values("cycle")

        features = engine_data[feature_columns].values
        targets = engine_data["RUL"].values

        for i in range(len(engine_data) - window_size + 1):

            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])

    return np.array(X), np.array(y)


X_train, y_train = create_windows(
    train_data, feature_columns, WINDOW_SIZE
)

X_val, y_val = create_windows(
    val_data, feature_columns, WINDOW_SIZE
)

print("Training windows:", X_train.shape)
print("Training targets:", y_train.shape)

print("Validation windows:", X_val.shape)
print("Validation targets:", y_val.shape)

Training windows: (14241, 30, 18)
Training targets: (14241,)
Validation windows: (3490, 30, 18)
Validation targets: (3490,)


In [13]:
# ==========================================
# STEP 10: CREATE PYTORCH DATALOADERS
# ==========================================

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Device:", device)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Device: mps
Training batches: 112
Validation batches: 28


In [14]:
# ==========================================
# STEP 11: PREPARE TEST SEQUENCES
# ==========================================

X_test, y_test = [], []

for unit in sorted(test_df["unit"].unique()):

    engine_data = test_df[
        test_df["unit"] == unit
    ].sort_values("cycle")

    # Take the last 30 observed cycles
    features = engine_data[feature_columns].values[-WINDOW_SIZE:]

    X_test.append(features)

X_test = np.array(X_test)

# Actual RUL labels for the 100 test engines
y_test = test_rul["RUL"].values

print("Test windows:", X_test.shape)
print("Test targets:", y_test.shape)

Test windows: (100, 30, 18)
Test targets: (100,)


In [15]:
# ==========================================
# STEP 12: SAVE PREPROCESSED DATASETS
# ==========================================

np.savez_compressed(
    "../data/processed/preprocessed_data.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("Preprocessed datasets saved successfully!")

Preprocessed datasets saved successfully!
